# Problem Set 9

[PSet 9](pset9.pdf)

##TODO: Add solutions for Problem Set 9 Examples 6 and 7

In [1]:
import sympy as sp
import sympy.physics.units as spu
import sympy.physics.mechanics as spm
import sympy.physics.vector as spv
from sympy.physics.vector.printing import init_vprinting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import IPython

# Import IPython display for proper LaTeX formatting
from IPython.display import display, Math, Markdown

# Initialize symbols
sp.init_printing()

# Enable dot notation printing for dynamicsymbols
init_vprinting(use_latex="mathjax")

HALF = sp.S.Half

In [2]:
def reference_frame(frame: str, x=r"\imath", y=r"\jmath", z=r"k") -> spm.ReferenceFrame:
    """Create a SymPy reference frame with custom basis vector labels.

    Parameters
    ----------
    frame : str
        The name of the reference frame.
    x, y, z : str
        Labels for the basis vectors.
    """
    return spm.ReferenceFrame(
        frame,
        latexs=(
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{x}}}",
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{y}}}",
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{z}}}",
        ),
    )


def reference_frame_circular(name: str, angle=r"theta") -> spm.ReferenceFrame:
    """Create a circular reference frame with radial and angular basis labels.

    Parameters
    ----------
    name : str
        Name of the new reference frame.
    angle : str, optional
        Symbol or label used for the angular basis vector, by default "theta".
    """
    return reference_frame(name, x=r"r", y=rf"\{angle}", z=r"e_z")

## Problem 9.1 Ping Pong

A ping-pong ball and a bowling ball collide elastically on a frictionless surface.
The magnitude of the initial velocity of the ping-pong ball is $v_{p,0}$ and the direction of the velocity is in the positive x-direction.
The magnitude of the initial velocity of the bowling ball is $v_{b,0}$ and the direction of the velocity is in the negative x-direction.

You may assume that the mass of the bowling ball is much greater than the mass 
of the ping-pong ball.

A small ping-pong ball moving to the right collides elastically with 
a much larger bowling ball moving to the left on a flat frictionless surface. 
The larger ball is heavier, and the scene is a simple physics illustration 
showing a light object striking a heavy object

![ping pong collision with bowling ball](../figures/PS0901-ping-pong.jpg)

After the collision, what is the component of the velocity of the ping-pong 
ball in the x direction? (pay attention to signs). 

Express your answer in some or all of the following variables: $v_{p,0}$ and $v_{b,0}$


In [3]:
(
    vp_0,  # Magnitude of initial velocity of ping pong
    vb_0,  # Magnitude of initial velocity of bowling ball
    mp,  # Mass of ping pong
    mb,  # mass of bowling ball
) = sp.symbols("v_{p0} v_{b0} m_p m_b ")

In [4]:
vp_1, vb_1 = sp.symbols("v_{p1} v_{b1}")

cmx = sp.Eq(mp * vp_0 - mb * vb_0, mp * vp_1 + mb * vb_1)
cme = sp.Eq(
    HALF * mp * vp_0**2 + HALF * mb * vb_0**2, HALF * mp * vp_1**2 + HALF * mb * vb_1**2
)

coll_sol = sp.solve([cmx, cme], [vb_1, vp_1], dict=True)[1][vp_1]
display(Math(rf"\text{{Velocity for ping pong ball: }} {spv.vlatex(coll_sol)}"))

display(
    Math(
        rf"\text{{Velocity for ping pong ball in the limit of heavy bowling ball}}: "
        rf"\boxed{{{spv.vlatex(coll_sol.limit(mb, sp.oo))}}}"
    )
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Problem 9.2 Satellite Fly-By
A satellite (with mass negligible compared to that of the Earth) is making a 
fly-by of the Earth on a hyperbolic orbit. Let the velocity of the Earth be
$\vec v_e = v_e \vec \imath$. The satellite originally approaches from the 
negative x-direction, goes around the earth, and then flies off in the 
same direction it came from. Far from the Earth, the magnitude of the 
satellite’s x-component of velocity is $v_i$ when approaching and $v_f$ 
when receding.

![satellite fly-by](../figures/PS0902-satellite-fly-by.jpg)

For the case where $\vec v_i = 4v_e \vec \imath$, what is the final speed 
$v_f$ of the satellite in terms of the speed of the earth $v_e$?

In [5]:
(
    m_s,
    m_e,
    v_si,
    v_sf,
    v_ei,
    v_ef,
) = sp.symbols(r"m_s m_e v_{s\,i} v_{s\,f} v_{e\,i} v_{e\,f}")

KE_i = HALF * m_s * v_si**2 + HALF * m_e * v_ei**2
KE_f = HALF * m_s * v_sf**2 + HALF * m_e * v_ef**2

Momentum_i = m_s * v_si + m_e * v_ei
Momentum_f = m_s * v_sf + m_e * v_ef

Energy_conserved = sp.Eq(KE_i, KE_f)
Momentum_conserved = sp.Eq(Momentum_i, Momentum_f)

vel_solutions = sp.solve(
    [Energy_conserved, Momentum_conserved], [v_sf, v_ef], dict=True
)[1]
vel_solutions

vel_sf = vel_solutions[v_sf]
vel_conditions = spm.msubs(vel_sf, {v_si: 4 * v_ei})
vel_solutions_limit = sp.Abs(sp.limit(vel_conditions, m_e, sp.oo))

display(
    Math(
        r"\text{{Final speed of satellite}} = "
        rf"\boxed{{{spm.vlatex(vel_solutions_limit)}}}"
    )
)

<IPython.core.display.Math object>

## Problem 9.3 Block Sliding up Curved Surface

A block of mass $m_b$ sits at rest on a frictionless table; the block has a 
circular surface of radius $R$ as shown in the figure. 
A small cube of mass $m_c < m_b$ and speed $v_{c,0}$ is
incident upon the block; the cube moves without friction on the table 
and the block.

Notice that in this problem, the block does move since it is not fixed to the
table

![Block Sliding up Curved Surface](../figures/PS0903-Block-Sliding.jpg)

(a) What is the maximum height h above the table that is reached by the cube?
Assume $v_{c,0}$ is small enough so that $h < R$. Express your answer using some or
all of the following variables: $v_{c,0}$, $g$, $m_b$ and $m_c$.

(b) After the cube reaches the maximum height, it slides back down the wedge. After
the two separate again, what is the speed (magnitude of the velocity) of each?

Express your answer using some or all of the following variables: $v_{c,0}$, $g$, $m_b$ and $m_c$.

In [6]:
# Conservation of momentum and collision analysis in the x-direction.
# Momentum is conserved in the block-cube system during the collision as
# there are no external forces acting in the x-direction.

(
    m_block,  # mass of the block
    R,  # radius of the circular surface
    m_cube,  # mass of the cube
    v_cube_i,  # initial speed of the cube
    g,  # acceleration due to gravity
) = sp.symbols(r"m_{block} R m_{cube} v_{cube\,i}, g")


In [7]:
# Kinematics of cube and block
t = spm.dynamicsymbols._t
theta = spm.dynamicsymbols("theta")
N = reference_frame("N")

# Geometry of the cube on the circular surface of fixed radius R.
R = sp.symbols("R", real=True, positive=True)

block_x = spm.dynamicsymbols("b_x")
pos_block_x = block_x * N.x
vel_block_x = pos_block_x.dt(N)
acc_block_x = vel_block_x.dt(N)

display(Markdown("### Kinematics of the block"))
display(Math(rf"\text{{Displacement of the block: }} {spv.vlatex(pos_block_x)}"))
display(Math(rf"\text{{Velocity of the block: }} {spv.vlatex(vel_block_x)}"))
display(Math(rf"\text{{Acceleration of the block: }} {spv.vlatex(acc_block_x)}"))

# The cube moves on a circular arc of fixed radius R. Its position is measured
# relative to the block, so the cube's center is at
#   x_c = x_b + R*sin(theta)
#   y_c = R*(1 - cos(theta))
# and therefore the vertical height is a consequence of the motion, not an
# independent variable imposed by R = h/(1 - cos(theta)).

cube_x = block_x + R * sp.sin(theta)
cube_y = R * (1 - sp.cos(theta))
h = cube_y  # instantaneous height above the table

cube_r = cube_x * N.x + cube_y * N.y
cube_vel = cube_r.dt(N)
cube_acc = cube_vel.dt(N)

display(Markdown("---\n### Kinematics of the cube"))
display(Math(rf"\text{{Displacement of the cube: }} {spv.vlatex(cube_r)}"))
display(Math(rf"\text{{Velocity of the cube: }} {spv.vlatex(cube_vel)}"))
display(Math(rf"\text{{Acceleration of the cube: }} {spv.vlatex(cube_acc.express(N))}"))

### Kinematics of the block

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

---
### Kinematics of the cube

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [8]:
# Collision of the cube with the block. The block is initially at rest.
# The block is on a frictionless table.

v_cube_block_collision = sp.Symbol("v_{cube\_block\_collision}", real=True)
momentum_conservation = sp.Eq(
    m_cube * v_cube_i, (m_cube + m_block) * v_cube_block_collision
)
v_cube_block_collision_sol = sp.solve(momentum_conservation, v_cube_block_collision)[0]

display(Markdown("### Conservation of Momentum of the cube with the block"))
display(
    Math(
        r"\text{Velocity of the cube and block immediately after collision: } "
        rf"\boxed{{{spv.vlatex(v_cube_block_collision_sol)}}}"
    )
)

### Conservation of Momentum of the cube with the block

<IPython.core.display.Math object>

In [9]:
display(Markdown("### a. Maximum height reached by the cube"))
# At maximum height, the cube and block move with the same horizontal velocity
# Momentum is conserved in the horizontal direction
# A maximum height speed is 0
# Using conservation of momentum and energy, we can solve for the maximum height

vel_block_cube = sp.Symbol("v_{block,cube}")

momentum_conservation_at_max_h = sp.Eq(
    m_cube * v_cube_i, (m_cube + m_block) * vel_block_cube
)
vel_block_cube_sol = sp.solve(momentum_conservation_at_max_h, vel_block_cube)[0]
display(
    Math(
        r"\text{Velocity of the block and cube at maximum height: } "
        + rf"{spv.vlatex(vel_block_cube_sol)}"
    )
)

h = sp.Symbol("h")
conservation_energy_at_max_h = sp.Eq(
    HALF * m_cube * v_cube_i**2,
    HALF * (m_cube + m_block) * vel_block_cube_sol**2 + m_cube * g * h,
)

h_sol = sp.solve(conservation_energy_at_max_h, h)[0].simplify()

display(
    Math(
        r"\text{Maximum height reached by the cube: } "
        rf"\; \boxed{{{spv.vlatex(h_sol)}}}"
    )
)

### a. Maximum height reached by the cube

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Problem 9.4 Spring Blocks Motion

![Spring Blocks Motion](../figures/PS0904-Spring-Block-Motion.jpg)

Block 1 of mass $m_1$ is moving with speed $v_1$ to the right on a horizontal frictionless table
as shown in the figure above. The block collides with the free end of a spring-block
system of $m_2$ that is initially at rest. The collision is one-dimensional. The spring has
spring constant $k$.
What is the maximum compression of the spring? Express your answer using some or
all of the following: $k$, $m_1$, $m_2$ and $v_1$.

In [10]:
(
    m1,  # Mass of block 1. Units: kg
    m2,  # Mass of block 2. Units: kg
    v1,  # Initial velocity of block 1. Units: m/s
    k,  # Spring constant. Units: N/m
) = sp.symbols("m_1 m_2 v_1 k", real=True, positive=True)

In [11]:
# At maximum compression of the spring, the blocks move with the same velocity.
# Using conservation of momentum:
v_f = sp.Symbol(
    "v_f", real=True
)  # final velocity of both blocks at maximum compression

p_i = m1 * v1
p_f = m1 * v_f + m2 * v_f
v_f_sol = sp.solve(p_i - p_f, v_f)[0]
display(
    Math(
        r"\text{Velocity of both blocks at first collision: } v_f ="
        rf"{spm.vlatex(v_f_sol)}"
    )
)

# Conservation of energy at maximum compression:
x = sp.Symbol("x", real=True, positive=True)
ke_i = HALF * m1 * v1**2
ke_f = HALF * m1 * v_f_sol**2 + HALF * m2 * v_f_sol**2
U_spring = HALF * k * x**2
CME = sp.Eq(ke_i, ke_f + U_spring)
x_solutions = sp.solve(CME, x)[0]
display(
    Math(
        rf"\text{{Maximum compression of the spring}} = "
        rf"{spm.vlatex(sp.sqrt(x_solutions**2, evaluate=False))}"
    )
)

<IPython.core.display.Math object>

<IPython.core.display.Math object>